In [ ]:
# In[1]:

import pandas as pd

# Load dataframes and parse timestamps as UTC datetimes (Unix seconds)
df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

df_log = pd.read_csv("log.csv")
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

df_error = pd.read_csv("error_logs.csv")
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates by (cmdb_id, kpi_name)
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(
        count='count',
        min_value='min',
        max_value='max',
        mean_value='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    )
    .sort_values('count', ascending=False)
)
# keep display compact: top 20
metric_agg_head = metric_agg.head(20)

# 2) trace.csv aggregates by (cmdb_id, trace_name)
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(
        count='count',
        min_value='min',
        max_value='max',
        mean_value='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    )
    .sort_values('count', ascending=False)
)
trace_agg_head = trace_agg.head(20)

# 3) log.csv aggregates by (cmdb_id, log_name)
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(
        count='count',
        min_value='min',
        max_value='max',
        mean_value='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    )
    .sort_values('count', ascending=False)
)
log_agg_head = log_agg.head(20)

# 4) error_logs.csv: unique cmdb_id list with count of messages per cmdb_id,
#    and overall earliest and latest timestamps in that file.
error_counts = df_error.groupby('cmdb_id', as_index=False).agg(count=('message','count')).sort_values('count', ascending=False)
error_time_range = pd.Series({
    'error_earliest_utc': df_error['timestamp'].min(),
    'error_latest_utc': df_error['timestamp'].max()
})

# 5) overall time range across all four files combined (UTC datetimes)
all_timestamps = pd.concat([
    df_metric['timestamp'],
    df_trace['timestamp'],
    df_log['timestamp'],
    df_error['timestamp']
], ignore_index=True)
overall_time_range = pd.Series({
    'overall_earliest_utc': all_timestamps.min(),
    'overall_latest_utc': all_timestamps.max()
})

# Display compact results (top20 tables and small summaries)
metric_agg_head, trace_agg_head, log_agg_head, error_counts.head(20), error_time_range, overall_time_range

```
Out[1]:
```
```python
# Reuse existing aggregated variables from previous run:
# metric_agg_head, trace_agg_head, log_agg_head, error_counts, error_time_range, overall_time_range

# Prepare short lists of notable metric KPIs to call out
notable_kpis = ['diskio', 'cpu', 'latency-90', 'latency-50', 'mem']
top_metrics = metric_agg_head[metric_agg_head['kpi_name'].isin(notable_kpis)].head(10)

# Prepare notable traces (look for traces from/to frontend/root for checkoutservice)
notable_traces = trace_agg_head[
    trace_agg_head['trace_name'].str.contains('from_frontend|from_root|to_checkoutservice|from_checkoutservice', regex=True, na=False)
].head(6)

# Frontend log error_count row if present
frontend_error_row = log_agg_head[
    (log_agg_head['cmdb_id'] == 'frontend') & (log_agg_head['log_name'] == 'log.error_count')
]
frontend_error_summary = None
if not frontend_error_row.empty:
    r = frontend_error_row.iloc[0]
    frontend_error_summary = f"frontend log.error_count: count={int(r['count'])}, max={r['max_value']:.1f}, mean={r['mean_value']:.1f}, p95={r['p95']:.1f}"

# Top error log counts (compact)
top_error_counts = error_counts.head(5)

# Build the plain-English summary
lines = []
lines.append(f"Overall telemetry time range (UTC): {overall_time_range['overall_earliest_utc']} to {overall_time_range['overall_latest_utc']}.")
lines.append(f"Error-logs time range (UTC): {error_time_range['error_earliest_utc']} to {error_time_range['error_latest_utc']}.")
lines.append("")  # separator

# Error log counts
lines.append("Error-log counts (top):")
for _, row in top_error_counts.iterrows():
    lines.append(f" - {row['cmdb_id']}: {int(row['count'])} messages")
lines.append("")

# Frontend logs quick note (if found)
if frontend_error_summary:
    lines.append("Frontend logs:")
    lines.append(" - " + frontend_error_summary)
    lines.append("")

# Notable metric observations
lines.append("Notable metric observations (compact):")
for _, r in top_metrics.iterrows():
    lines.append(
        f" - {r['cmdb_id']} / {r['kpi_name']}: count={int(r['count'])}, max={r['max_value']:.3g}, mean={r['mean_value']:.3g}, p95={r['p95']:.3g}"
    )
lines.append("")

# Notable trace observations
if not notable_traces.empty:
    lines.append("Notable trace observations (compact):")
    for _, r in notable_traces.iterrows():
        lines.append(
            f" - {r['cmdb_id']} / {r['trace_name']}: count={int(r['count'])}, max={r['max_value']:.3g}, mean={r['mean_value']:.3g}, p95={r['p95']:.3g}"
        )
    lines.append("")

lines.append("Summary conclusion:")
lines.append(" - The frontend has the largest number of error-log messages by far (top count shown).")
lines.append(" - Metrics show high disk I/O and memory values for adservice, CPU and memory spikes for cartservice, and elevated latencies for checkoutservice (see compact lines above).")
lines.append(" - Traces for checkoutservice indicate notable latency when interacting with frontend (elevated mean and p95).")
lines.append(" - Use the lines above for quick triage: investigate frontend error logs, adservice disk I/O, cartservice CPU/memory, and checkoutservice latency paths.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  count     min_value     max_value    mean_value           p50           p95
0           adservice         cpu     25  3.453743e-01  7.322929e-01  6.117382e-01  6.382325e-01  7.261072e-01
1           adservice      diskio     25  3.997884e+04  1.910377e+05  8.025739e+04  6.682994e+04  1.526733e+05
2           adservice  latency-50     25  2.956376e-03  4.007931e-03  3.200080e-03  2.988890e-03  3.786119e-03
3           adservice  latency-90     25  4.599931e-03  1.222737e-02  6.100420e-03  4.626085e-03  1.017552e-02
4           adservice         mem     25  1.074347e+08  1.096671e+08  1.083164e+08  1.077604e+08  1.095430e+08
5           adservice      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
6           adservice    workload     25  4.975567e+00  1.496220e+01  1.223204e+01  1.372008e+01  1.448136e+01
7         cartservice         cpu     25  6.230739e-01  7.341304e+00  2.128406e+00  1.889884e+00  5.617402e+00
9         cartservice  latency-50     25  3.083286e-03  6.040914e-03  3.377886e-03  3.154684e-03  5.237722e-03
41  frontend-external       error     25  0.000000e+00  6.466667e+00  4.854053e-01  0.000000e+00  3.768110e+00
10        cartservice  latency-90     25  4.749916e-03  3.837006e-02  8.812461e-03  4.878431e-03  3.555278e-02
11        cartservice         mem     25  5.083866e+07  8.769812e+07  5.317893e+07  5.094762e+07  6.624566e+07
12        cartservice      socket     25  4.000000e+00  8.000000e+00  4.261333e+00  4.000000e+00  6.026667e+00
13        cartservice    workload     25  1.020665e+01  2.477811e+01  2.059299e+01  2.306313e+01  2.425426e+01
14    checkoutservice         cpu     25  1.550408e-01  4.466922e-01  3.380766e-01  3.404214e-01  4.285082e-01
15    checkoutservice  latency-50     25  7.647564e-02  1.243866e+00  3.684768e-01  2.876533e-01  8.873475e-01
16    checkoutservice  latency-90     25  4.518189e-01  2.740968e+00  8.811539e-01  6.967917e-01  1.899620e+00
17    checkoutservice         mem     25  1.087379e+07  1.154785e+07  1.120680e+07  1.119409e+07  1.151260e+07
18    checkoutservice      socket     25  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00
19    checkoutservice    workload     25  4.392727e-01  1.355556e+00  7.860150e-01  7.976333e-01  1.055110e+00,             cmdb_id                                trace_name  count  min_value   max_value  mean_value         p50         p95
0   checkoutservice  trace.from_checkoutservice.duration_mean     25   0.003462    0.015811    0.005632    0.003848    0.012685
1   checkoutservice   trace.from_checkoutservice.duration_p95     25   0.005382    0.024452    0.007656    0.006340    0.013514
2   checkoutservice     trace.from_checkoutservice.error_rate     25   0.000000    0.265060    0.011791    0.000000    0.023762
3   checkoutservice      trace.from_checkoutservice.row_count     25  47.000000  299.000000  178.120000  188.000000  288.000000
4   checkoutservice  trace.from_frontendservice.duration_mean     25   0.057069    1.035526    0.373211    0.319584    0.906883
5   checkoutservice   trace.from_frontendservice.duration_p95     25   0.314393    3.129083    0.818600    0.623090    1.937706
6   checkoutservice     trace.from_frontendservice.error_rate     25   0.000000    0.818182    0.081534    0.000000    0.653050
7   checkoutservice      trace.from_frontendservice.row_count     25   6.000000   35.000000   22.600000   23.000000   33.800000
8   checkoutservice             trace.from_root.duration_mean     25   0.022721    0.184745    0.070696    0.059607    0.135860
9   checkoutservice              trace.from_root.duration_p95     25   0.097137    0.591780    0.181537    0.138153    0.373927
10  checkoutservice                trace.from_root.error_rate     25   0.000000    0.402985    0.017810    0.000000    0.033803
11  checkoutservice                 trace.from_root.row_count     25  21.000000  153.000000   97.440000  102.000000  143.200000
12  checkoutservice    trace.to_checkoutservice.duration_mean     25   0.003462    0.015811    0.005632    0.003848    0.012685
13  checkoutservice     trace.to_checkoutservice.duration_p95     25   0.005382    0.024452    0.007656    0.006340    0.013514
14  checkoutservice       trace.to_checkoutservice.error_rate     25   0.000000    0.265060    0.011791    0.000000    0.023762
15  checkoutservice        trace.to_checkoutservice.row_count     25  47.000000  299.000000  178.120000  188.000000  288.000000
16  checkoutservice    trace.to_currencyservice.duration_mean     25   0.000083    0.003481    0.001138    0.001001    0.002902
17  checkoutservice     trace.to_currencyservice.duration_p95     25   0.000147    0.000537    0.000244    0.000214    0.000391
18  checkoutservice       trace.to_currencyservice.error_rate     25   0.000000    0.000000    0.000000    0.000000    0.000000
19  checkoutservice        trace.to_currencyservice.row_count     25  16.000000  129.000000   73.200000   78.000000  119.200000,                   cmdb_id         log_name  count  min_value  max_value   mean_value     p50     p95
0               adservice  log.error_count     25          0          0     0.000000     0.0     0.0
1               adservice    log.row_count     25         59        450   339.800000   404.0   433.6
2             cartservice  log.error_count     25          0          0     0.000000     0.0     0.0
3             cartservice    log.row_count     25        109        769   588.400000   697.0   745.6
4         checkoutservice  log.error_count     25          0          0     0.000000     0.0     0.0
5         checkoutservice    log.row_count     25         18        105    63.160000    66.0    99.6
6         currencyservice  log.error_count     25          0          0     0.000000     0.0     0.0
7         currencyservice    log.row_count     25        264       2401  1838.400000  2141.0  2371.4
8            emailservice  log.error_count     25          0          0     0.000000     0.0     0.0
9            emailservice    log.row_count     25          6         35    20.280000    21.0    33.2
10               frontend  log.error_count     25          0        637   131.920000     0.0   549.2
11               frontend    log.row_count     25        373       2998  2326.320000  2480.0  2970.2
12         paymentservice  log.error_count     25          0          0     0.000000     0.0     0.0
13         paymentservice    log.row_count     25         12         70    40.680000    42.0    66.4
20        shippingservice  log.error_count     25          0          0     0.000000     0.0     0.0
21        shippingservice    log.row_count     25         72        498   339.160000   380.0   462.0
17  recommendationservice    log.row_count     23          4        555   368.695652   502.0   548.9
16  recommendationservice  log.error_count     23          0          0     0.000000     0.0     0.0
19                  redis    log.row_count      5          5          5     5.000000     5.0     5.0
18                  redis  log.error_count      5          0          0     0.000000     0.0     0.0,     cmdb_id  count
0  frontend   3298, error_earliest_utc   2024-01-15 19:34:08+00:00
error_latest_utc     2024-01-15 19:45:47+00:00
dtype: datetime64[ns, UTC], overall_earliest_utc   2024-01-15 19:21:00+00:00
overall_latest_utc     2024-01-15 19:45:47+00:00
dtype: datetime64[ns, UTC])```
```

In [ ]:
# In[2]:

# Reuse existing df_metric if available; otherwise load and parse timestamps
try:
    df_metric
except NameError:
    import pandas as pd
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

import pandas as pd

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-15 19:18:51+00:00", utc=True)
window_end = pd.to_datetime("2024-01-15 19:48:51+00:00", utc=True)

# Filter metric series to candidate services for threshold calculation
df_candidates = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Rule: calculate global p95 thresholds on the entire series BEFORE window filtering
thresholds = (
    df_candidates
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95_threshold'})
)

# Filter rows to the incident window (still only candidate services)
df_window = df_candidates[
    (df_candidates['timestamp'] >= window_start) &
    (df_candidates['timestamp'] <= window_end)
].copy()

# If there are no points in the window, result should be empty
if df_window.empty:
    anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','p95_threshold','anomaly_count',
        'earliest_anomaly_utc','max_value_in_window','mean_value_in_window','total_points_in_window'
    ])
else:
    # Merge thresholds into window rows so each row knows its global p95 threshold
    merged = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # Mark anomalies where value >= global p95 threshold
    merged['is_anom'] = merged['value'] >= merged['p95_threshold']

    # Aggregate per series within the window
    grouped = merged.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        total_points_in_window = ('value','size'),
        max_value_in_window = ('value','max'),
        mean_value_in_window = ('value','mean'),
        anomaly_count = ('is_anom','sum')
    )

    # Earliest anomaly timestamp per series (if any)
    earliest = (
        merged[merged['is_anom']]
        .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anomaly_utc'})
    )

    # Combine aggregates with earliest anomaly and the precomputed threshold
    anomaly_summary = (
        grouped
        .merge(earliest, on=['cmdb_id','kpi_name'], how='left')
        .merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    )

    # Reorder and format columns as requested
    anomaly_summary = anomaly_summary[[
        'cmdb_id','kpi_name','p95_threshold','anomaly_count','earliest_anomaly_utc',
        'max_value_in_window','mean_value_in_window','total_points_in_window'
    ]]

    # Sort by anomaly_count descending and keep top 20
    anomaly_summary = anomaly_summary.sort_values('anomaly_count', ascending=False).head(20)

# Display compact table (top 20)
anomaly_summary

```
Out[2]:
```
Summary (plain English, UTC):

Overall
- Analysis restricted to the five candidate services in the given incident window (2024-01-15 19:18:51 to 2024-01-15 19:48:51). Most series have 25 points in-window (recommendationservice.diskio has 21).

Notable findings by service
- checkoutservice
  - Multiple KPIs exceeded their global P95 at least briefly (anomaly_count = 2 for several meaningful KPIs): cpu, latency-50, latency-90, mem, workload.
  - Earliest anomaly times (UTC): workload 2024-01-15 19:21:00, cpu 19:22:00, latency-50 19:42:00, latency-90 19:44:00, mem 19:43:00.
  - Max values in-window show the spikes (e.g., cpu max 0.4467, latency-50 max 1.2439, latency-90 max 2.7409).

- currencyservice
  - Several KPIs show brief P95 exceedances (anomaly_count = 2): cpu, latency-50, latency-90, mem, workload, socket.
  - Earliest anomaly times (UTC): cpu 19:23:00, latency-50 19:41:00, latency-90 19:41:00, mem 19:42:00, workload 19:28:00, socket 19:42:00.
  - Max values in-window indicate spikes (e.g., cpu max ~18.90, mem max ~8.02e7, workload max ~80.20).

- emailservice
  - Brief P95 exceedances (anomaly_count = 2) for cpu, latency-50, latency-90, mem.
  - Earliest anomaly times (UTC): cpu 19:22:00, latency-50 19:42:00, latency-90 19:42:00, mem 19:44:00.
  - emailservice.socket and some diskio/error series have p95_threshold = 0 or constant values causing anomaly_count to equal total points — these should be interpreted cautiously.

- productcatalogservice
  - Not present in the compact result (no series with points exceeding global P95 in-window shown), so no P95 anomalies reported here.

- recommendationservice
  - Only diskio appears and has p95_threshold = 0 with 21 points in-window (anomaly_count = total points); treat this as not informative.

Caveats
- Several series have global P95 thresholds equal to 0 (e.g., error counters, some diskio) or constant values (socket counts). These produce anomaly_count equal to total points in-window and are not useful signals by themselves — interpret them cautiously.
- The meaningful anomalies (non-zero thresholds) are short-lived (mostly 2 points in a 25-point window) but present across checkoutservice and currencyservice (and to a lesser extent emailservice).

Recommended next steps
- Prioritize triage of checkoutservice and currencyservice: investigate the endpoints/traces that produced the latency and CPU/memory spikes around the timestamps called out above (19:21–19:44 UTC).
- Inspect application logs and traces for those services at the earliest anomaly times to identify root causes.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  p95_threshold  anomaly_count      earliest_anomaly_utc  max_value_in_window  mean_value_in_window  total_points_in_window
7         currencyservice       error   0.000000e+00             25 2024-01-15 19:21:00+00:00         0.000000e+00          0.000000e+00                      25
4         checkoutservice      socket   9.000000e+00             25 2024-01-15 19:21:00+00:00         9.000000e+00          9.000000e+00                      25
14           emailservice      diskio   0.000000e+00             25 2024-01-15 19:21:00+00:00         0.000000e+00          0.000000e+00                      25
18           emailservice      socket   3.000000e+00             25 2024-01-15 19:21:00+00:00         3.000000e+00          3.000000e+00                      25
28  recommendationservice      diskio   0.000000e+00             21 2024-01-15 19:21:00+00:00         0.000000e+00          0.000000e+00                      21
0         checkoutservice         cpu   4.285082e-01              2 2024-01-15 19:22:00+00:00         4.466922e-01          3.380766e-01                      25
1         checkoutservice  latency-50   8.873475e-01              2 2024-01-15 19:42:00+00:00         1.243866e+00          3.684768e-01                      25
6         currencyservice         cpu   1.881317e+01              2 2024-01-15 19:23:00+00:00         1.890431e+01          1.619018e+01                      25
5         checkoutservice    workload   1.055110e+00              2 2024-01-15 19:21:00+00:00         1.355556e+00          7.860150e-01                      25
3         checkoutservice         mem   1.151260e+07              2 2024-01-15 19:43:00+00:00         1.154785e+07          1.120680e+07                      25
9         currencyservice  latency-90   4.184548e-01              2 2024-01-15 19:41:00+00:00         4.903130e-01          1.519105e-01                      25
10        currencyservice         mem   7.864223e+07              2 2024-01-15 19:42:00+00:00         8.016282e+07          4.727486e+07                      25
12        currencyservice    workload   7.816298e+01              2 2024-01-15 19:28:00+00:00         8.019998e+01          6.457767e+01                      25
11        currencyservice      socket   6.910000e+00              2 2024-01-15 19:42:00+00:00         7.000000e+00          3.607333e+00                      25
13           emailservice         cpu   2.912652e-01              2 2024-01-15 19:22:00+00:00         3.012337e-01          2.613654e-01                      25
15           emailservice  latency-50   3.133787e-03              2 2024-01-15 19:42:00+00:00         3.208468e-03          3.019332e-03                      25
8         currencyservice  latency-50   1.621499e-01              2 2024-01-15 19:41:00+00:00         1.704744e-01          7.275171e-02                      25
2         checkoutservice  latency-90   1.899620e+00              2 2024-01-15 19:44:00+00:00         2.740968e+00          8.811539e-01                      25
17           emailservice         mem   4.367292e+07              2 2024-01-15 19:44:00+00:00         4.367809e+07          4.358687e+07                      25
16           emailservice  latency-90   5.330767e-03              2 2024-01-15 19:42:00+00:00         5.729600e-03          4.689469e-03                      25```
```

In [ ]:
# In[3]:

# Reuse existing df_trace if available; otherwise load and parse timestamps as required
try:
    df_trace
except NameError:
    import pandas as pd
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
else:
    # Ensure timestamp is parsed as UTC datetime (in case df_trace was loaded without parsing)
    if not pd.api.types.is_datetime64_any_dtype(df_trace['timestamp']):
        import pandas as pd
        df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

import pandas as pd

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-15 19:18:51+00:00", utc=True)
window_end = pd.to_datetime("2024-01-15 19:48:51+00:00", utc=True)

# Restrict to candidate services for threshold computation
df_candidates = df_trace[df_trace['cmdb_id'].isin(candidates)].copy()

# Compute global P95 thresholds per (cmdb_id, trace_name) BEFORE window filtering
thresholds = (
    df_candidates
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'p95_threshold'})
)

# Filter trace rows to the incident window (still limited to candidate services)
df_window = df_candidates[
    (df_candidates['timestamp'] >= window_start) &
    (df_candidates['timestamp'] <= window_end)
].copy()

# If no rows in window, return empty DataFrame with requested schema
if df_window.empty:
    trace_anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','p95_threshold','anomaly_count','earliest_anomaly_utc',
        'max_value_in_window','mean_value_in_window','total_points_in_window'
    ])
else:
    # Merge thresholds into window rows
    merged = df_window.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

    # Mark anomalies where value >= global p95 threshold
    merged['is_anom'] = merged['value'] >= merged['p95_threshold']

    # Aggregate per series within the window
    grouped = merged.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        total_points_in_window = ('value','size'),
        max_value_in_window = ('value','max'),
        mean_value_in_window = ('value','mean'),
        anomaly_count = ('is_anom','sum')
    )

    # Earliest anomaly timestamp per series (if any)
    earliest = (
        merged[merged['is_anom']]
        .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anomaly_utc'})
    )

    # Combine aggregates with earliest anomaly and thresholds
    trace_anomaly_summary = (
        grouped
        .merge(earliest, on=['cmdb_id','trace_name'], how='left')
        .merge(thresholds, on=['cmdb_id','trace_name'], how='left')
    )

    # Reorder columns as requested
    trace_anomaly_summary = trace_anomaly_summary[[
        'cmdb_id','trace_name','p95_threshold','anomaly_count','earliest_anomaly_utc',
        'max_value_in_window','mean_value_in_window','total_points_in_window'
    ]]

    # Sort by anomaly_count desc then by max_value_in_window desc, limit to top 20
    trace_anomaly_summary = trace_anomaly_summary.sort_values(
        by=['anomaly_count','max_value_in_window'],
        ascending=[False, False]
    ).head(20)

# Display the compact table (variable name as final output)
trace_anomaly_summary

```
Out[3]:
```
Summary (plain English, UTC):

Incident window: 2024-01-15 19:18:51 to 2024-01-15 19:48:51.

Key findings
- Most trace "error_rate" series for these services have a global P95 of 0.0, and therefore every in-window point meets or exceeds that threshold. Those series (e.g., trace.to_currencyservice.error_rate, trace.to_emailservice.error_rate, trace.from_frontendservice.error_rate, many recommendationservice error_rate metrics) are not useful signals here because the P95=0 makes anomaly_count = total points.
- The most meaningful trace anomalies are in the row_count series:
  - emailservice / trace.from_root.row_count — p95_threshold = 24.0, anomaly_count = 23, earliest anomaly 2024-01-15 19:22:00 UTC, max_in_window = 24, mean_in_window = 23.04, 25 points in-window.
  - currencyservice / trace.from_root.row_count — p95_threshold = 12.0, anomaly_count = 22, earliest anomaly 2024-01-15 19:22:00 UTC, max_in_window = 12, mean_in_window = 11.16, 25 points in-window.

Interpretation and recommendation
- The row_count series indicate sustained higher-than-normal request counts to emailservice and currencyservice starting around 19:22 UTC. These are likely load spikes and are actionable signals.
- The many zero-valued p95 error_rate series should be treated cautiously (they produce false-positive-like anomaly counts). Do not prioritize them without supporting evidence (latency, logs, metrics).
- Next steps: investigate traces and logs for emailservice and currencyservice around ~19:22 UTC (look for increased request volume, latency, queueing, or downstream contention). Correlate with metric anomalies (CPU/memory/latency) found earlier to identify root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  p95_threshold  anomaly_count      earliest_anomaly_utc  max_value_in_window  mean_value_in_window  total_points_in_window
18        checkoutservice          trace.to_currencyservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
22        checkoutservice             trace.to_emailservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
26        checkoutservice           trace.to_paymentservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
30        checkoutservice    trace.to_productcatalogservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
34        currencyservice        trace.from_checkoutservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
38        currencyservice        trace.from_frontendservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
42        currencyservice                   trace.from_root.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
46           emailservice        trace.from_checkoutservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
50           emailservice                   trace.from_root.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
54  productcatalogservice        trace.from_checkoutservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
58  productcatalogservice        trace.from_frontendservice.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
66  productcatalogservice                   trace.from_root.error_rate            0.0             25 2024-01-15 19:21:00+00:00                  0.0                  0.00                      25
51           emailservice                    trace.from_root.row_count           24.0             23 2024-01-15 19:22:00+00:00                 24.0                 23.04                      25
70  recommendationservice        trace.from_frontendservice.error_rate            0.0             23 2024-01-15 19:21:00+00:00                  0.0                  0.00                      23
74  recommendationservice  trace.from_recommendationservice.error_rate            0.0             23 2024-01-15 19:21:00+00:00                  0.0                  0.00                      23
78  recommendationservice                   trace.from_root.error_rate            0.0             23 2024-01-15 19:21:00+00:00                  0.0                  0.00                      23
86  recommendationservice    trace.to_recommendationservice.error_rate            0.0             23 2024-01-15 19:21:00+00:00                  0.0                  0.00                      23
43        currencyservice                    trace.from_root.row_count           12.0             22 2024-01-15 19:22:00+00:00                 12.0                 11.16                      25
62  productcatalogservice  trace.from_recommendationservice.error_rate            0.0             22 2024-01-15 19:21:00+00:00                  0.0                  0.00                      22
82  recommendationservice    trace.to_productcatalogservice.error_rate            0.0             22 2024-01-15 19:21:00+00:00                  0.0                  0.00                      22```
```

In [ ]:
# In[4]:

# Load and parse files if not already in memory; parse timestamps as UTC datetimes
import pandas as pd

try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(df_log['timestamp']):
        df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")
    df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(df_error['timestamp']):
        df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Candidate services and incident window (UTC)
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]
window_start = pd.to_datetime("2024-01-15 19:18:51+00:00", utc=True)
window_end   = pd.to_datetime("2024-01-15 19:48:51+00:00", utc=True)

# 1) log.csv: restrict to candidates and incident window
df_log_window = df_log[
    (df_log['timestamp'] >= window_start) &
    (df_log['timestamp'] <= window_end) &
    (df_log['cmdb_id'].isin(candidates))
].copy()

if df_log_window.empty:
    log_summary = pd.DataFrame(columns=[
        'cmdb_id','log_name','count_in_window','min_value','max_value','mean_value','p50_value','p95_value'
    ])
else:
    grp = df_log_window.groupby(['cmdb_id','log_name'])['value']
    agg_basic = grp.agg(count_in_window='count', min_value='min', max_value='max', mean_value='mean').reset_index()
    # compute quantiles (p50 and p95)
    q = grp.quantile([0.5,0.95]).unstack(level=-1).reset_index()
    # columns: ['cmdb_id','log_name',0.5,0.95] - rename
    q = q.rename(columns={0.5:'p50_value', 0.95:'p95_value'})
    log_summary = agg_basic.merge(q, on=['cmdb_id','log_name'], how='left')
    # sort by count_in_window desc and keep top 20
    log_summary = log_summary.sort_values('count_in_window', ascending=False).head(20)

# 2) error_logs.csv: counts and earliest/latest for the five cmdb_ids
df_error_window = df_error[
    (df_error['timestamp'] >= window_start) &
    (df_error['timestamp'] <= window_end)
].copy()

# Aggregate counts and min/max timestamps per cmdb_id within window
if df_error_window.empty:
    error_agg = pd.DataFrame(columns=['cmdb_id','count_messages_in_window','earliest_message_utc','latest_message_utc'])
else:
    error_agg_raw = df_error_window.groupby('cmdb_id', as_index=False).agg(
        count_messages_in_window=('message','count'),
        earliest_message_utc=('timestamp','min'),
        latest_message_utc=('timestamp','max')
    )
    # Ensure we return rows for the five candidate services (zero/NaT if none)
    error_agg = pd.DataFrame({'cmdb_id': candidates})
    error_agg = error_agg.merge(error_agg_raw, on='cmdb_id', how='left')
    error_agg['count_messages_in_window'] = error_agg['count_messages_in_window'].fillna(0).astype(int)
    # earliest/latest remain NaT where no messages

# 3) up to 10 example messages from error_logs.csv within window for the candidates
if df_error_window.empty:
    error_examples = pd.DataFrame(columns=['cmdb_id','timestamp_utc','message'])
else:
    error_examples = df_error_window[df_error_window['cmdb_id'].isin(candidates)][
        ['cmdb_id','timestamp','message']
    ].sort_values('timestamp', ascending=True).head(10).rename(columns={'timestamp':'timestamp_utc'})

# Final compact outputs (variables shown as final results)
log_summary, error_agg, error_examples

```
Out[4]:
```
Summary (plain English, UTC):

Incident window: 2024-01-15 19:18:51 to 2024-01-15 19:48:51.

Findings from logs (candidate services only)
- No error-log messages for any of the five candidate services in the incident window (checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice).
- log.error_count is zero across all series for these services during the window.
- log.row_count (request volume) in-window:
  - currencyservice: 25 points, mean ~1838 requests, p95 ~2371 — sustained high traffic.
  - recommendationservice: 23 points, mean ~369, p95 ~549 — moderately high and variable traffic.
  - checkoutservice: 25 points, mean ~63, p95 ~99 — moderate traffic.
  - emailservice: 25 points, mean ~20, p95 ~33 — low traffic.
  - productcatalogservice: only 1 point in-window (value 3) — minimal data available.

Error logs examples
- There are zero error_log messages for these five services in the window (no example messages to show).

Interpretation and recommendation
- There are no application error messages for the candidate services during the incident window, so the issue is unlikely to be captured as explicit error-log entries for these services.
- The most notable signal is high request volume for currencyservice (very large mean and p95 row_count) and elevated volume/variability for recommendationservice. These services warrant priority investigation (check traces, resource metrics, and downstream dependencies around ~19:22–19:45 UTC).
- productcatalogservice has almost no log data in-window — treat its absence of signals cautiously (could be low traffic or missing telemetry).

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  count_in_window  min_value  max_value   mean_value  p50_value  p95_value
0        checkoutservice  log.error_count               25          0          0     0.000000        0.0        0.0
1        checkoutservice    log.row_count               25         18        105    63.160000       66.0       99.6
2        currencyservice  log.error_count               25          0          0     0.000000        0.0        0.0
3        currencyservice    log.row_count               25        264       2401  1838.400000     2141.0     2371.4
4           emailservice  log.error_count               25          0          0     0.000000        0.0        0.0
5           emailservice    log.row_count               25          6         35    20.280000       21.0       33.2
9  recommendationservice    log.row_count               23          4        555   368.695652      502.0      548.9
8  recommendationservice  log.error_count               23          0          0     0.000000        0.0        0.0
7  productcatalogservice    log.row_count                1          3          3     3.000000        3.0        3.0
6  productcatalogservice  log.error_count                1          0          0     0.000000        0.0        0.0,                  cmdb_id  count_messages_in_window earliest_message_utc latest_message_utc
0        checkoutservice                         0                  NaT                NaT
1        currencyservice                         0                  NaT                NaT
2           emailservice                         0                  NaT                NaT
3  productcatalogservice                         0                  NaT                NaT
4  recommendationservice                         0                  NaT                NaT, Empty DataFrame
Columns: [cmdb_id, timestamp_utc, message]
Index: [])```
```

In [ ]:
# In[5]:

# Load metric.csv if not already loaded; parse timestamp as UTC datetimes
import pandas as pd
import numpy as np

try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(df_metric['timestamp']):
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Candidate services and incident window (UTC)
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]
window_start = pd.to_datetime("2024-01-15 19:18:51+00:00", utc=True)
window_end   = pd.to_datetime("2024-01-15 19:48:51+00:00", utc=True)

# Restrict metric series to candidate services for threshold calculation (global p95)
df_candidates = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Rule: compute global p95 thresholds BEFORE filtering by window
thresholds = (
    df_candidates
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'p95_threshold'})
)

# Filter metric rows to the incident window
df_window = df_candidates[
    (df_candidates['timestamp'] >= window_start) &
    (df_candidates['timestamp'] <= window_end)
].copy()

# Merge thresholds into window rows
merged = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Identify anomaly rows where value >= global p95_threshold
merged['is_anom'] = merged['value'] >= merged['p95_threshold']

# Keep only anomaly rows for fault grouping
anom = merged[merged['is_anom']].sort_values(['cmdb_id','kpi_name','timestamp']).copy()

if anom.empty:
    faults_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_utc','fault_end_utc','length_points',
        'max_value_in_fault','p95_threshold','severity_ratio','earliest_anomaly_utc'
    ])
else:
    # For each (cmdb_id, kpi_name), group consecutive minutes into faults
    def group_consecutive(df):
        df = df.sort_values('timestamp').copy()
        # time difference between consecutive rows
        td = df['timestamp'].diff()
        # start new fault where diff > 1 minute or first row
        new_fault = (td > pd.Timedelta(minutes=1)) | td.isna()
        fault_seq = new_fault.cumsum()  # cumulative to identify fault groups
        df = df.assign(fault_seq=fault_seq)
        return df

    anom_grouped = anom.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(group_consecutive)

    # Aggregate per fault (cmdb_id, kpi_name, fault_seq)
    agg = (
        anom_grouped
        .groupby(['cmdb_id','kpi_name','fault_seq'], as_index=False)
        .agg(
            fault_start_utc = ('timestamp','first'),
            fault_end_utc = ('timestamp','last'),
            length_points = ('timestamp','size'),
            max_value_in_fault = ('value','max'),
            p95_threshold = ('p95_threshold','first')
        )
    )

    # severity_ratio: max_value_in_fault / p95_threshold, set NaN if p95_threshold == 0 or NaN
    agg['severity_ratio'] = agg.apply(
        lambda r: (r['max_value_in_fault'] / r['p95_threshold']) if (pd.notna(r['p95_threshold']) and r['p95_threshold'] != 0) else np.nan,
        axis=1
    )

    # earliest_anomaly_utc is same as fault_start_utc
    agg['earliest_anomaly_utc'] = agg['fault_start_utc']

    # Select and order columns
    faults_summary = agg[[
        'cmdb_id','kpi_name','fault_start_utc','fault_end_utc','length_points',
        'max_value_in_fault','p95_threshold','severity_ratio','earliest_anomaly_utc'
    ]].copy()

    # Sort by severity_ratio desc then by length_points desc, limit to top 20
    faults_summary = faults_summary.sort_values(['severity_ratio','length_points'], ascending=[False, False]).head(20)

# Final compact table output
faults_summary

```
Out[5]:
```
Summary (plain English, UTC):

Incident window: 2024-01-15 19:18:51 to 2024-01-15 19:48:51.

Top faults (compact findings)
- recommendationservice — cpu: strongest fault. Two consecutive anomaly minutes from 19:21 to 19:22 UTC; max value ~12.25 vs global p95 ~3.47 (severity_ratio ≈ 3.53). Highest priority.
- recommendationservice — latency-50: single-point spike at 19:43 UTC; max ~17.40 vs p95 ~9.48 (severity_ratio ≈ 1.83).
- checkoutservice — latency-90: two consecutive anomaly minutes 19:44–19:45 UTC; max ~2.74 vs p95 ~1.90 (severity_ratio ≈ 1.44).
- productcatalogservice — socket: two-minute anomaly 19:44–19:45 UTC; max 8.0 vs p95 5.6 (severity_ratio ≈ 1.43).
- checkoutservice — latency-50 and checkoutservice/workload/mem also show short anomalies (mostly 1–2 points) with severity_ratio in the ~1.3–1.4 range.
- emailservice — workload and latency-90 show brief anomalies around 19:21–19:43 UTC (short, lower severity ratios ~1.04–1.26).
- currencyservice — several single-point latency anomalies at ~19:41–19:44 UTC with modest severity (>1.04).

General observations and interpretation
- The strongest, most actionable signal is the short but large CPU spike on recommendationservice at ~19:21–19:22 UTC (severity > 3× the series P95). This is the highest-severity fault in the window.
- Most other anomalies are short (1–2 minutes) and have modest severity (severity_ratio between ~1.05 and ~1.44). They may be transient but worth correlating.
- Faults span multiple services (recommendationservice, checkoutservice, productcatalogservice, emailservice, currencyservice), but recommendationservice CPU stands out.

Recommended next steps
1. Immediately investigate recommendationservice CPU around 19:21–19:22 UTC: check traces, logs, recent deployments, and resource (CPU/memory) metrics and thread/worker saturation.
2. Correlate the short latency faults (checkoutservice, recommendationservice, productcatalogservice) with traces and downstream calls to see whether they stem from the recommendationservice spike or from external load.
3. Check request volume (row_count) and any orchestration/instance scaling events at ~19:20–19:45 UTC to see if traffic surges or instance restarts align with these faults.
4. Treat single-minute anomalies as transient unless they reoccur; prioritize remediation for multi-minute or high-severity (recommendationservice CPU) events.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name           fault_start_utc             fault_end_utc  length_points  max_value_in_fault  p95_threshold  severity_ratio      earliest_anomaly_utc
40  recommendationservice         cpu 2024-01-15 19:21:00+00:00 2024-01-15 19:22:00+00:00              2        1.224932e+01   3.472535e+00        3.527488 2024-01-15 19:21:00+00:00
44  recommendationservice  latency-50 2024-01-15 19:43:00+00:00 2024-01-15 19:43:00+00:00              1        1.739699e+01   9.484179e+00        1.834316 2024-01-15 19:43:00+00:00
4         checkoutservice  latency-90 2024-01-15 19:44:00+00:00 2024-01-15 19:45:00+00:00              2        2.740968e+00   1.899620e+00        1.442904 2024-01-15 19:44:00+00:00
37  productcatalogservice      socket 2024-01-15 19:44:00+00:00 2024-01-15 19:45:00+00:00              2        8.000000e+00   5.600000e+00        1.428571 2024-01-15 19:44:00+00:00
3         checkoutservice  latency-50 2024-01-15 19:44:00+00:00 2024-01-15 19:44:00+00:00              1        1.243866e+00   8.873475e-01        1.401781 2024-01-15 19:44:00+00:00
36  productcatalogservice         mem 2024-01-15 19:44:00+00:00 2024-01-15 19:45:00+00:00              2        1.879639e+07   1.355880e+07        1.386287 2024-01-15 19:44:00+00:00
7         checkoutservice    workload 2024-01-15 19:21:00+00:00 2024-01-15 19:21:00+00:00              1        1.355556e+00   1.055110e+00        1.284753 2024-01-15 19:21:00+00:00
27           emailservice    workload 2024-01-15 19:21:00+00:00 2024-01-15 19:21:00+00:00              1        1.355556e+00   1.076887e+00        1.258773 2024-01-15 19:21:00+00:00
34  productcatalogservice  latency-90 2024-01-15 19:36:00+00:00 2024-01-15 19:36:00+00:00              1        8.136102e-03   6.631378e-03        1.226910 2024-01-15 19:36:00+00:00
46  recommendationservice  latency-90 2024-01-15 19:43:00+00:00 2024-01-15 19:43:00+00:00              1        2.729472e+01   2.279398e+01        1.197453 2024-01-15 19:43:00+00:00
15        currencyservice  latency-90 2024-01-15 19:44:00+00:00 2024-01-15 19:44:00+00:00              1        4.903130e-01   4.184548e-01        1.171723 2024-01-15 19:44:00+00:00
31  productcatalogservice       error 2024-01-15 19:39:00+00:00 2024-01-15 19:39:00+00:00              1        3.000000e-01   2.600000e-01        1.153846 2024-01-15 19:39:00+00:00
43  recommendationservice  latency-50 2024-01-15 19:39:00+00:00 2024-01-15 19:39:00+00:00              1        1.025221e+01   9.484179e+00        1.080980 2024-01-15 19:39:00+00:00
24           emailservice  latency-90 2024-01-15 19:42:00+00:00 2024-01-15 19:43:00+00:00              2        5.729600e-03   5.330767e-03        1.074817 2024-01-15 19:42:00+00:00
45  recommendationservice  latency-90 2024-01-15 19:39:00+00:00 2024-01-15 19:39:00+00:00              1        2.399874e+01   2.279398e+01        1.052854 2024-01-15 19:39:00+00:00
13        currencyservice  latency-50 2024-01-15 19:44:00+00:00 2024-01-15 19:44:00+00:00              1        1.704744e-01   1.621499e-01        1.051338 2024-01-15 19:44:00+00:00
12        currencyservice  latency-50 2024-01-15 19:41:00+00:00 2024-01-15 19:41:00+00:00              1        1.699616e-01   1.621499e-01        1.048176 2024-01-15 19:41:00+00:00
29  productcatalogservice         cpu 2024-01-15 19:25:00+00:00 2024-01-15 19:25:00+00:00              1        2.922224e+00   2.790410e+00        1.047238 2024-01-15 19:25:00+00:00
1         checkoutservice         cpu 2024-01-15 19:30:00+00:00 2024-01-15 19:30:00+00:00              1        4.466922e-01   4.285082e-01        1.042436 2024-01-15 19:30:00+00:00
21           emailservice         cpu 2024-01-15 19:34:00+00:00 2024-01-15 19:34:00+00:00              1        3.012337e-01   2.912652e-01        1.034225 2024-01-15 19:34:00+00:00```
```

In [ ]:
# In[6]:

import pandas as pd

# Load df_trace if not in memory and ensure timestamp is parsed as UTC datetimes
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(df_trace['timestamp']):
        df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-15 19:18:51+00:00", utc=True)
window_end   = pd.to_datetime("2024-01-15 19:48:51+00:00", utc=True)

# Candidate services (case-sensitive substring match)
candidates = [
    "recommendationservice",
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice"
]
pattern = "|".join(candidates)

# Filter traces to incident window
df_trace_window = df_trace[
    (df_trace['timestamp'] >= window_start) &
    (df_trace['timestamp'] <= window_end)
].copy()

# Keep only rows where cmdb_id or trace_name references any candidate (case-sensitive substring)
mask = (
    df_trace_window['cmdb_id'].astype(str).str.contains(pattern, na=False) |
    df_trace_window['trace_name'].astype(str).str.contains(pattern, na=False)
)
df_trace_candidates = df_trace_window[mask].copy()

# Group by (cmdb_id, trace_name) and compute aggregates
if df_trace_candidates.empty:
    trace_agg_head = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_in_window','earliest_timestamp_utc',
        'min_value','max_value','mean_value','p95_value'
    ])
else:
    grp = df_trace_candidates.groupby(['cmdb_id','trace_name'], as_index=False)['value']
    trace_agg = grp.agg(
        count_in_window = 'count',
        min_value = 'min',
        max_value = 'max',
        mean_value = 'mean'
    )
    # compute earliest timestamp and p95 separately
    earliest = df_trace_candidates.groupby(['cmdb_id','trace_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_timestamp_utc'})
    p95 = grp.quantile(0.95).reset_index().rename(columns={'value':'p95_value'})
    # merge
    trace_agg = trace_agg.merge(earliest, on=['cmdb_id','trace_name'], how='left').merge(p95, on=['cmdb_id','trace_name'], how='left')
    # sort and keep top 20
    trace_agg_head = trace_agg.sort_values('count_in_window', ascending=False).head(20)

# Up to 10 raw trace rows involving recommendationservice (cmdb_id == 'recommendationservice' OR trace_name contains it)
if df_trace_window.empty:
    rec_examples = pd.DataFrame(columns=['timestamp_utc','cmdb_id','trace_name','value'])
else:
    rec_mask = (
        (df_trace_window['cmdb_id'] == 'recommendationservice') |
        (df_trace_window['trace_name'].astype(str).str.contains('recommendationservice', na=False))
    )
    rec_examples = df_trace_window[rec_mask].copy()
    if not rec_examples.empty:
        rec_examples = rec_examples[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=True).head(10).rename(columns={'timestamp':'timestamp_utc'})

# Final compact outputs
trace_agg_head, rec_examples

```
Out[6]:
```
Summary (plain English, UTC):

Incident window: 2024-01-15 19:18:51 → 2024-01-15 19:48:51.

Key observations from filtered traces
- Checkoutservice dominates the trace traffic in-window (many trace series with 25 points each).
  - High request counts to checkoutservice: trace.from_checkoutservice.row_count and trace.to_checkoutservice.row_count show max ~299, mean ~178, p95 ~288 — sustained, high per-minute traffic.
  - Latency signals for checkoutservice are notable: trace.from_frontendservice.duration_mean has mean ~0.373s and p95 ≈ 0.907s; duration_p95 max ~3.13s in-window (p95 value ~1.94).
- Recommendationservice shows very large local activity at 19:21 UTC:
  - Examples include recommendationservice trace.to_recommendationservice.row_count = 78 at 19:21 and recommendationservice trace.to_recommendationservice.duration_mean ≈ 0.0031 at the same minute. Several rows at 19:21 reference recommendationservice (both as origin and destination).
- Many trace error_rate series have p95 = 0.0 (so they are not useful here); focus instead on row_count and duration series for signal.
- All candidate services have earliest in-window traces at 2024-01-15 19:21:00 UTC (the window is well-covered starting that minute).

Actionable takeaways
1. Investigate recommendationservice around 19:21 UTC because of the large row_count (78) and multiple trace entries at that time — this is the clearest spike in the trace examples.
2. Correlate checkoutservice high row_count and observed frontend→checkout latencies (p95 up to ~3s) with recommendationservice activity and upstream traffic — checkoutservice shows both high volume and meaningful latency.
3. Ignore zero-valued error_rate p95 series as noisy; prioritize row_count and duration p95 spikes plus corresponding logs/metrics around 19:21–19:45 UTC.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                trace_name  count_in_window  min_value   max_value  mean_value    earliest_timestamp_utc  index   p95_value
0   checkoutservice  trace.from_checkoutservice.duration_mean               25   0.003462    0.015811    0.005632 2024-01-15 19:21:00+00:00      0    0.012685
1   checkoutservice   trace.from_checkoutservice.duration_p95               25   0.005382    0.024452    0.007656 2024-01-15 19:21:00+00:00      1    0.013514
2   checkoutservice     trace.from_checkoutservice.error_rate               25   0.000000    0.265060    0.011791 2024-01-15 19:21:00+00:00      2    0.023762
3   checkoutservice      trace.from_checkoutservice.row_count               25  47.000000  299.000000  178.120000 2024-01-15 19:21:00+00:00      3  288.000000
4   checkoutservice  trace.from_frontendservice.duration_mean               25   0.057069    1.035526    0.373211 2024-01-15 19:21:00+00:00      4    0.906883
5   checkoutservice   trace.from_frontendservice.duration_p95               25   0.314393    3.129083    0.818600 2024-01-15 19:21:00+00:00      5    1.937706
6   checkoutservice     trace.from_frontendservice.error_rate               25   0.000000    0.818182    0.081534 2024-01-15 19:21:00+00:00      6    0.653050
7   checkoutservice      trace.from_frontendservice.row_count               25   6.000000   35.000000   22.600000 2024-01-15 19:21:00+00:00      7   33.800000
8   checkoutservice             trace.from_root.duration_mean               25   0.022721    0.184745    0.070696 2024-01-15 19:21:00+00:00      8    0.135860
9   checkoutservice              trace.from_root.duration_p95               25   0.097137    0.591780    0.181537 2024-01-15 19:21:00+00:00      9    0.373927
10  checkoutservice                trace.from_root.error_rate               25   0.000000    0.402985    0.017810 2024-01-15 19:21:00+00:00     10    0.033803
11  checkoutservice                 trace.from_root.row_count               25  21.000000  153.000000   97.440000 2024-01-15 19:21:00+00:00     11  143.200000
12  checkoutservice    trace.to_checkoutservice.duration_mean               25   0.003462    0.015811    0.005632 2024-01-15 19:21:00+00:00     12    0.012685
13  checkoutservice     trace.to_checkoutservice.duration_p95               25   0.005382    0.024452    0.007656 2024-01-15 19:21:00+00:00     13    0.013514
14  checkoutservice       trace.to_checkoutservice.error_rate               25   0.000000    0.265060    0.011791 2024-01-15 19:21:00+00:00     14    0.023762
15  checkoutservice        trace.to_checkoutservice.row_count               25  47.000000  299.000000  178.120000 2024-01-15 19:21:00+00:00     15  288.000000
16  checkoutservice    trace.to_currencyservice.duration_mean               25   0.000083    0.003481    0.001138 2024-01-15 19:21:00+00:00     16    0.002902
17  checkoutservice     trace.to_currencyservice.duration_p95               25   0.000147    0.000537    0.000244 2024-01-15 19:21:00+00:00     17    0.000391
18  checkoutservice       trace.to_currencyservice.error_rate               25   0.000000    0.000000    0.000000 2024-01-15 19:21:00+00:00     18    0.000000
19  checkoutservice        trace.to_currencyservice.row_count               25  16.000000  129.000000   73.200000 2024-01-15 19:21:00+00:00     19  119.200000,                 timestamp_utc                cmdb_id                                    trace_name      value
76  2024-01-15 19:21:00+00:00        frontendservice  trace.to_recommendationservice.duration_mean   0.003577
151 2024-01-15 19:21:00+00:00                   root      trace.to_recommendationservice.row_count   4.000000
150 2024-01-15 19:21:00+00:00                   root     trace.to_recommendationservice.error_rate   0.000000
149 2024-01-15 19:21:00+00:00                   root   trace.to_recommendationservice.duration_p95   0.000040
148 2024-01-15 19:21:00+00:00                   root  trace.to_recommendationservice.duration_mean   0.000033
123 2024-01-15 19:21:00+00:00  recommendationservice      trace.to_recommendationservice.row_count  78.000000
122 2024-01-15 19:21:00+00:00  recommendationservice     trace.to_recommendationservice.error_rate   0.000000
121 2024-01-15 19:21:00+00:00  recommendationservice   trace.to_recommendationservice.duration_p95   0.004249
120 2024-01-15 19:21:00+00:00  recommendationservice  trace.to_recommendationservice.duration_mean   0.003093
118 2024-01-15 19:21:00+00:00  recommendationservice     trace.to_productcatalogservice.error_rate   0.000000)```
```